# Portfolio construction and valuation

**Prerequisites:** complete **03** (market data and curves) and **05** (pricing fundamentals: `MarketContext`, instrument JSON, valuation flow) in this curriculum. This notebook builds a **portfolio spec** as JSON, wires **discounting** through a shared curve, and runs the **portfolio** parse → build → value → cashflow ladder pipeline.

## Concepts: portfolios as data, not objects

A **portfolio** here is a JSON **`PortfolioSpec`**: identifiers, base currency, **entities** (holders), and **positions**. Each position points at an **`instrument_spec`** (tagged JSON: `type` + `spec`)—the same shape you already use for single-instrument pricing.

The typical flow is:

1. **`parse_portfolio_spec`** — validate and canonicalize the spec string.
2. **`build_portfolio_from_spec`** — load into the engine and round-trip the spec (useful for debugging wire format).
3. **`value_portfolio`** — PV each position, convert to **base currency**, and return a **`PortfolioValuation`** JSON snapshot (per-position values, entity subtotals, **total** in base ccy).
4. **`aggregate_metrics`** — given that valuation JSON plus market and `as_of`, compute **portfolio-level metric aggregation** (summable risks, FX-aware where applicable).
5. **`aggregate_full_cashflows`** — merge contractual cashflows into a **date ladder** (`by_date`, `by_position`).

**Wire-format note:** `value_portfolio` returns **`PortfolioValuation`** only. Helpers **`portfolio_result_total_value`** and **`portfolio_result_get_metric`** expect a full **`PortfolioResult`** envelope (`valuation` + `metrics` + `meta`). The walkthrough shows both: read the total from the valuation JSON, then wrap a minimal `PortfolioResult` to exercise the helpers.

### Imports

Portfolio entry points live in `finstack_quant.portfolio`; curves and **`MarketContext`** come from `finstack_quant.core.market_data` (see notebook **03**).

In [1]:
import json
from datetime import date

from finstack_quant.core.market_data import DiscountCurve, MarketContext
from finstack_quant.portfolio import (
    aggregate_full_cashflows,
    aggregate_metrics,
    build_portfolio_from_spec,
    parse_portfolio_spec,
    portfolio_result_get_metric,
    portfolio_result_total_value,
    value_portfolio,
)

print("Imported portfolio pipeline and market helpers.")

Imported portfolio pipeline and market helpers.


### Market context: shared discount curve

Insert a **`DiscountCurve`** into **`MarketContext`**, then call **`to_json()`** so the same snapshot is passed into **`value_portfolio`**, **`aggregate_metrics`**, and **`aggregate_full_cashflows`**.

In [2]:
mc = MarketContext()
curve = DiscountCurve(
    "USD-OIS",
    date(2025, 1, 15),
    [
        (0.0, 1.0),
        (0.25, 0.9875),
        (0.5, 0.975),
        (1.0, 0.95),
        (2.0, 0.90),
        (5.0, 0.75),
    ],
    day_count="act_365f",
)
mc.insert(curve)
market_json = mc.to_json()
curves = json.loads(market_json).get("curves", [])
curve_ids = [c.get("id") for c in curves if isinstance(c, dict)]
print("Market OK; curve ids in snapshot:", curve_ids)

Market OK; curve ids in snapshot: ['USD-OIS']


### Portfolio spec: two USD deposits

Each **position** references **`discount_curve_id`** that must exist in **`market_json`**. **`quantity`** scales the economic exposure (here `1.0` **units**).

In [3]:
portfolio = json.dumps(
    {
        "id": "credit-portfolio",
        "as_of": "2025-01-15",
        "base_ccy": "USD",
        "entities": {"ENTITY-1": {"id": "ENTITY-1"}},
        "positions": [
            {
                "position_id": "POS-1",
                "entity_id": "ENTITY-1",
                "instrument_id": "DEP-3M",
                "instrument_spec": {
                    "type": "deposit",
                    "spec": {
                        "id": "DEP-3M",
                        "notional": {"amount": 1_000_000.0, "currency": "USD"},
                        "start_date": "2025-01-15",
                        "maturity": "2025-04-15",
                        "day_count": "Act360",
                        "quote_rate": 0.045,
                        "discount_curve_id": "USD-OIS",
                        "attributes": {},
                    },
                },
                "quantity": 1.0,
                "unit": "units",
            },
            {
                "position_id": "POS-2",
                "entity_id": "ENTITY-1",
                "instrument_id": "DEP-6M",
                "instrument_spec": {
                    "type": "deposit",
                    "spec": {
                        "id": "DEP-6M",
                        "notional": {"amount": 2_000_000.0, "currency": "USD"},
                        "start_date": "2025-01-15",
                        "maturity": "2025-07-15",
                        "day_count": "Act360",
                        "quote_rate": 0.05,
                        "discount_curve_id": "USD-OIS",
                        "attributes": {},
                    },
                },
                "quantity": 1.0,
                "unit": "units",
            },
        ],
    }
)
spec_obj = json.loads(portfolio)
print("Spec id:", spec_obj["id"], "| positions:", len(spec_obj["positions"]))
print("Position ids:", [p["position_id"] for p in spec_obj["positions"]])

Spec id: credit-portfolio | positions: 2
Position ids: ['POS-1', 'POS-2']


### `parse_portfolio_spec`

Parse and canonicalize JSON. Failures raise **`ValueError`** with a Rust error string—fix the spec and retry.

In [4]:
canonical = parse_portfolio_spec(portfolio)
positions = json.loads(canonical).get("positions", [])
print("Parsed OK; canonical position count:", len(positions))
print("First instrument_id:", positions[0]["instrument_id"])

Parsed OK; canonical position count: 2
First instrument_id: DEP-3M


### `build_portfolio_from_spec`

Materialize the portfolio and return the spec after **`Portfolio::from_spec` → `to_spec`**. Use this to confirm instruments deserialize cleanly.

In [5]:
built = build_portfolio_from_spec(portfolio)
print("Round-trip prefix:", built[:200])
print("Full JSON length (chars):", len(built))

Round-trip prefix: {"id":"credit-portfolio","name":null,"base_ccy":"USD","as_of":"2025-01-15","entities":{"ENTITY-1":{"id":"ENTITY-1","name":null}},"positions":[{"position_id":"POS-1","entity_id":"ENTITY-1","instrument_
Full JSON length (chars): 393


### `value_portfolio`

Returns **`PortfolioValuation`** JSON. The **base-currency total** is under **`total_base_ccy`** (money object: `amount`, `currency`).

**PV sign convention (deposits).** For a **deposit** from the lender’s perspective, contractual cash flows are typically modeled as an initial **outflow** (funding the loan) and a terminal **inflow** (repayment plus interest). The engine’s **present value** is the sum of those discounted flows, so a **negative PV** means the position is economically a **loan / funding** from the holder’s viewpoint (money out today dominates discounted money in later). A **positive PV** would correspond to the opposite side of the trade (borrowing). When reading `position_values`, interpret the sign together with your **position quantity** and **book convention**.

In [6]:
val_result = value_portfolio(portfolio, market_json)
val_obj = json.loads(val_result)
print("Valuation keys:", list(val_obj.keys()))
print("Snippet:\n", json.dumps(val_obj, indent=2)[:900], "...")
total_amt = float(val_obj["total_base_ccy"]["amount"])
total_ccy = val_obj["total_base_ccy"]["currency"]
print(f"Total PV (from valuation JSON): {total_amt:,.6f} {total_ccy}")

Valuation keys: ['as_of', 'position_values', 'total_base_ccy', 'by_entity', 'fx_collapse_policy']
Snippet:
 {
  "as_of": "2025-01-15",
  "position_values": {
    "POS-1": {
      "position_id": "POS-1",
      "entity_id": "ENTITY-1",
      "value_native": {
        "amount": "998782.543494608",
        "currency": "USD"
      },
      "value_base": {
        "amount": "998782.543494608",
        "currency": "USD"
      },
      "metric_scale": 1.0,
      "risk_metrics_complete": true,
      "valuation_result": {
        "schema_version": 1,
        "instrument_id": "DEP-3M",
        "as_of": "2025-01-15",
        "value": {
          "amount": "998782.54349460794750000000",
          "currency": "USD"
        },
        "measures": {
          "theta": 136.82384572096635,
          "dv01": -24.62751477357233,
          "bucketed_dv01": -24.627514773688745,
          "bucketed_dv01::USD-OIS::10y": 0.0,
          "bucketed_dv01::USD-OIS::15y": 0.0,
          "bucketed_dv01::USD-OIS::1y":

### `portfolio_result_total_value` and `portfolio_result_get_metric`

These read a **`PortfolioResult`** envelope. Below we wrap the valuation dict with minimal **`metrics`** and **`meta`** so the helpers have a valid document. In production, reporting layers may emit this full shape for you.

In [7]:
meta = {
    "numeric_mode": "F64",
    "rounding": {
        "mode": "Bankers",
        "ingest_scale_by_ccy": {},
        "output_scale_by_ccy": {},
        "version": 1,
        "tolerances": {"generic_epsilon": 1e-10, "rate_epsilon": 1e-12},
    },
}
pv_total = float(val_obj["total_base_ccy"]["amount"])
metrics_payload = {
    "aggregated": {
        "pv": {
            "metric_id": "pv",
            "total": pv_total,
            "by_entity": {"ENTITY-1": pv_total},
        }
    },
    "by_position": {},
}
portfolio_result_json = json.dumps(
    {"valuation": val_obj, "metrics": metrics_payload, "meta": meta}
)
print("portfolio_result_total_value:", portfolio_result_total_value(portfolio_result_json))
print("portfolio_result_get_metric('pv'):", portfolio_result_get_metric(portfolio_result_json, "pv"))
print("portfolio_result_get_metric('dv01'):", portfolio_result_get_metric(portfolio_result_json, "dv01"))

portfolio_result_total_value: 2998224.711537264
portfolio_result_get_metric('pv'): 2998224.711537264
portfolio_result_get_metric('dv01'): None


### `aggregate_metrics`

Pass the **`value_portfolio`** JSON string, base currency, market JSON, and **`as_of`**. *Current limitation:* the valuation JSON from **`value_portfolio`** does not rehydrate per-position risk measures, so aggregation from this round-trip may return **empty** `aggregated` / `by_position` until the wire format includes them. The call pattern below is still the supported API.

In [8]:
agg = aggregate_metrics(val_result, "USD", market_json, "2025-01-15")
agg_obj = json.loads(agg)
print(json.dumps(agg_obj, indent=2))
print("Aggregated metric ids:", list(agg_obj.get("aggregated", {}).keys()))

{
  "aggregated": {
    "theta": {
      "metric_id": "theta",
      "total": 410.728679670603,
      "by_entity": {
        "ENTITY-1": 410.728679670603
      }
    },
    "dv01": {
      "metric_id": "dv01",
      "total": -123.77793465420837,
      "by_entity": {
        "ENTITY-1": -123.77793465420837
      }
    },
    "bucketed_dv01": {
      "metric_id": "bucketed_dv01",
      "total": -123.77793465432478,
      "by_entity": {
        "ENTITY-1": -123.77793465432478
      }
    },
    "bucketed_dv01::USD-OIS::10y": {
      "metric_id": "bucketed_dv01::USD-OIS::10y",
      "total": 0.0,
      "by_entity": {
        "ENTITY-1": 0.0
      }
    },
    "bucketed_dv01::USD-OIS::15y": {
      "metric_id": "bucketed_dv01::USD-OIS::15y",
      "total": 0.0,
      "by_entity": {
        "ENTITY-1": 0.0
      }
    },
    "bucketed_dv01::USD-OIS::1y": {
      "metric_id": "bucketed_dv01::USD-OIS::1y",
      "total": 0.5299307866953313,
      "by_entity": {
        "ENTITY-1": 0.5299307866

### `aggregate_full_cashflows`

Returns a typed **`PortfolioCashflows`** wrapper. Call **`to_json()`** for the full wire payload, or use its typed accessors for drill-downs. Amounts serialize as decimal strings on the wire.

In [9]:
cfs = aggregate_full_cashflows(portfolio, market_json)
cf_obj = json.loads(cfs.to_json())
print(json.dumps(cf_obj, indent=2)[:1400])
print("... [truncated if long] ...")
print("Ladder dates:", sorted(cf_obj.get("by_date", {}).keys()))

{
  "events": [
    {
      "position_id": "POS-1",
      "instrument_id": "DEP-3M",
      "instrument_type": "Deposit",
      "date": "2025-01-15",
      "amount": {
        "amount": "-1000000",
        "currency": "USD"
      },
      "kind": "Notional",
      "reset_date": null,
      "accrual_factor": 0.0,
      "rate": null
    },
    {
      "position_id": "POS-2",
      "instrument_id": "DEP-6M",
      "instrument_type": "Deposit",
      "date": "2025-01-15",
      "amount": {
        "amount": "-2000000",
        "currency": "USD"
      },
      "kind": "Notional",
      "reset_date": null,
      "accrual_factor": 0.0,
      "rate": null
    },
    {
      "position_id": "POS-1",
      "instrument_id": "DEP-3M",
      "instrument_type": "Deposit",
      "date": "2025-04-15",
      "amount": {
        "amount": "1011250",
        "currency": "USD"
      },
      "kind": "Fixed",
      "reset_date": null,
      "accrual_factor": 0.25,
      "rate": 0.045
    },
    {
      "posi

## Mini-example: three deposits (3M, 6M, 1Y)

Same curve and `as_of`. Different notionals and quoted rates. We print **total PV**, **`aggregate_metrics`** (may be empty), and a readable **cashflow ladder** sorted by date.

In [10]:
def deposit_position(
    position_id: str,
    instrument_id: str,
    notional: float,
    quote_rate: float,
    maturity: str,
) -> dict:
    return {
        "position_id": position_id,
        "entity_id": "ENTITY-1",
        "instrument_id": instrument_id,
        "instrument_spec": {
            "type": "deposit",
            "spec": {
                "id": instrument_id,
                "notional": {"amount": notional, "currency": "USD"},
                "start_date": "2025-01-15",
                "maturity": maturity,
                "day_count": "Act360",
                "quote_rate": quote_rate,
                "discount_curve_id": "USD-OIS",
                "attributes": {},
            },
        },
        "quantity": 1.0,
        "unit": "units",
    }


three_deposit_portfolio = json.dumps(
    {
        "id": "rates-ladder-book",
        "as_of": "2025-01-15",
        "base_ccy": "USD",
        "entities": {"ENTITY-1": {"id": "ENTITY-1"}},
        "positions": [
            deposit_position("L-3M", "DEP-3M-ME", 1_000_000.0, 0.045, "2025-04-15"),
            deposit_position("L-6M", "DEP-6M-ME", 2_000_000.0, 0.050, "2025-07-15"),
            deposit_position("L-1Y", "DEP-1Y-ME", 1_500_000.0, 0.055, "2026-01-15"),
        ],
    }
)

mini_val = value_portfolio(three_deposit_portfolio, market_json)
mini_v = json.loads(mini_val)
mini_total = float(mini_v["total_base_ccy"]["amount"])
print(f"Three-deposit total PV: {mini_total:,.6f} USD")

mini_agg = aggregate_metrics(mini_val, "USD", market_json, "2025-01-15")
print("aggregate_metrics:", mini_agg)

mini_cf = json.loads(aggregate_full_cashflows(three_deposit_portfolio, market_json).to_json())
by_date = mini_cf.get("by_date", {})
print("Cashflow ladder (date -> USD amount, summed across cashflow kinds):")
for d in sorted(by_date.keys()):
    usd_by_kind = by_date[d].get("USD", {})
    amt = sum(float(money["amount"]) for money in usd_by_kind.values())
    print(f"  {d}: {amt:,.2f} USD")
print("Done.")

Three-deposit total PV: 4,502,688.253204 USD
aggregate_metrics: {"aggregated":{"theta":{"metric_id":"theta","total":616.826081811334,"by_entity":{"ENTITY-1":616.826081811334}},"dv01":{"metric_id":"dv01","total":-274.22428907168796,"by_entity":{"ENTITY-1":-274.22428907168796}},"bucketed_dv01":{"metric_id":"bucketed_dv01","total":-274.2242890718044,"by_entity":{"ENTITY-1":-274.2242890718044}},"bucketed_dv01::USD-OIS::10y":{"metric_id":"bucketed_dv01::USD-OIS::10y","total":0.0,"by_entity":{"ENTITY-1":0.0}},"bucketed_dv01::USD-OIS::15y":{"metric_id":"bucketed_dv01::USD-OIS::15y","total":0.0,"by_entity":{"ENTITY-1":0.0}},"bucketed_dv01::USD-OIS::1y":{"metric_id":"bucketed_dv01::USD-OIS::1y","total":-149.91642363078427,"by_entity":{"ENTITY-1":-149.91642363078427}},"bucketed_dv01::USD-OIS::20y":{"metric_id":"bucketed_dv01::USD-OIS::20y","total":0.0,"by_entity":{"ENTITY-1":0.0}},"bucketed_dv01::USD-OIS::2y":{"metric_id":"bucketed_dv01::USD-OIS::2y","total":0.0,"by_entity":{"ENTITY-1":0.0}},"bu

## Performance (TWRR / MWRR) and Brinson attribution

Portfolio-level returns and attribution are available via lightweight JSON helpers. These operate on simple period/cashflow or sector-return records and do not require a full `Portfolio` object.


In [11]:
import json
from finstack_quant.portfolio import (
    twrr_linked,
    mwr_xirr,
    brinson_fachler,
    carino_link,
)

# Link sub-period returns (TWRR linking)
linked = twrr_linked(json.dumps([0.03, 0.025, 0.04]), 1.0)
print("twrr_linked (annualized):", linked)

# MWR (XIRR)
cf = json.dumps([
    {"date": "2025-01-01", "amount": -1_000_000.0},
    {"date": "2025-06-30", "amount": 50_000.0},
    {"date": "2026-01-01", "amount": 1_080_000.0},
])
x = mwr_xirr(cf)
print("mwr_xirr:", round(x, 6))

# Brinson-Fachler (single period)
sector_rows = [
    {"sector": "Tech", "portfolio_weight": 0.6, "benchmark_weight": 0.5, "portfolio_return": 0.04, "benchmark_return": 0.03},
    {"sector": "Fin",  "portfolio_weight": 0.4, "benchmark_weight": 0.5, "portfolio_return": 0.01, "benchmark_return": 0.015},
]
sectors = json.dumps(sector_rows)
bf = json.loads(brinson_fachler(sectors))
print("brinson_fachler excess:", round(bf.get("total_excess_return", 0), 6))

# Carino link (multi-period)
periods = json.dumps([sector_rows, sector_rows])  # toy reuse
cl = json.loads(carino_link(periods))
print("carino_link excess:", round(cl.get("total_excess_return", 0), 6))


twrr_linked (annualized): {"cumulative":0.09797999999999996,"annualised":0.09797999999999996,"num_periods":3}
mwr_xirr: 0.133273
brinson_fachler excess: 0.0055
carino_link excess: 0


## Takeaways

- A **portfolio** is **`PortfolioSpec` JSON**: entities + positions with embedded **`instrument_spec`** blocks.
- **`MarketContext.to_json()`** is the shared **market snapshot** for value, metrics, and cashflows.
- **`value_portfolio`** produces **`PortfolioValuation`** JSON; **total PV** is **`total_base_ccy.amount`** (parse with **`float`**).
- **`portfolio_result_*`** helpers expect a **`PortfolioResult`** envelope (**`valuation` + `metrics` + `meta`**).
- **`aggregate_full_cashflows`** gives a **date-indexed ladder** suitable for liquidity and P&L timing views.
- **`aggregate_metrics`** is the right hook for **summed risks** once valuation payloads carry per-position measures through JSON (or when called from native Rust with a live valuation).